# LeetCode 1365 — How Many Numbers Are Smaller Than the Current Number bằng Python

**Link đề:** https://leetcode.com/problems/how-many-numbers-are-smaller-than-the-current-number/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sorting, Counting Sort

```python
class Solution:
    def smallerNumbersThanCurrent(self, nums: list[int]) -> list[int]:
        ...
```

### Mục tiêu học
1. Viết lời giải trực giác hai vòng lặp `O(n²)` và hiểu vì sao nó chậm.
2. Nhận ra: "số phần tử nhỏ hơn `x`" = **vị trí đầu tiên của `x` trong mảng đã sắp xếp**.
3. Học **đếm tần suất + cộng dồn (prefix sum)** — kỹ thuật lõi của *counting sort* — để đạt `O(n + V)`.
4. Dùng `sorted`, `dict`, `enumerate`, `bisect` một cách có chủ đích.
5. Trace, chứng minh, so sánh 4 cách, test đối chiếu và tự giải biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho mảng `nums`. Với mỗi `nums[i]`, đếm có bao nhiêu `j` với `j != i` và `nums[j] < nums[i]`. Trả về mảng kết quả.

| nums | Output |
|---|---|
| `[8,1,2,2,3]` | `[4,0,1,1,3]` |
| `[6,5,4,8]` | `[2,1,0,3]` |
| `[7,7,7,7]` | `[0,0,0,0]` |

Giải thích ví dụ 1: số `8` lớn hơn bốn số `1, 2, 2, 3` → 4; số `1` không lớn hơn ai → 0; mỗi số `2` chỉ lớn hơn số `1` → 1 (hai số `2` **không** tính nhau vì điều kiện là `<` nghiêm ngặt); số `3` lớn hơn `1, 2, 2` → 3.

**Constraints:** `2 <= nums.length <= 500`, `0 <= nums[i] <= 100`.

Hai chi tiết quan trọng: (1) so sánh **nhỏ hơn hẳn** (`<`), (2) giá trị chỉ trong `[0, 100]` — gợi ý đếm tần suất.

In [1]:
nums1 = [8, 1, 2, 2, 3]
nums2 = [6, 5, 4, 8]
nums3 = [7, 7, 7, 7]
x = 3
print("Các số nhỏ hơn", x, "trong", nums1, ":", [v for v in nums1 if v < x])

Các số nhỏ hơn 3 trong [8, 1, 2, 2, 3] : [1, 2, 2]


Cell trên dùng **list comprehension có điều kiện**: `[v for v in nums1 if v < x]` — "lấy mọi `v` trong `nums1` mà `v < x`". Độ dài list đó chính là đáp án cho `x`.

Lưu ý: điều kiện `j != i` tự động thoả vì `nums[i] < nums[i]` luôn sai — chính phần tử `i` không bao giờ bị đếm.

## 2. Cách 1 — Brute Force hai vòng lặp

Với mỗi `i`, duyệt mọi `j`, đếm `nums[j] < nums[i]`.

In [2]:
def smaller_brute(nums):
    n = len(nums)
    result = []
    for i in range(n):
        count = 0
        for j in range(n):
            if j != i and nums[j] < nums[i]:
                count += 1
        result.append(count)
    return result

print(smaller_brute(nums1))   # [4, 0, 1, 1, 3]
print(smaller_brute(nums2))   # [2, 1, 0, 3]
print(smaller_brute(nums3))   # [0, 0, 0, 0]

[4, 0, 1, 1, 3]
[2, 1, 0, 3]
[0, 0, 0, 0]


Bản ngắn gọn với `sum` + generator:

In [3]:
def smaller_brute_short(nums):
    return [sum(v < x for v in nums) for x in nums]

print(smaller_brute_short(nums1))

[4, 0, 1, 1, 3]


### Độ phức tạp và nút thắt

- Thời gian: `O(n²)` — `n` lần, mỗi lần duyệt `n` phần tử. Với `n = 500` là 250.000 phép so sánh: vẫn qua LeetCode.
- Bộ nhớ: `O(1)` ngoài mảng kết quả.

**Nút thắt:** với hai số bằng nhau (ví dụ hai số `2`), ta đếm lại **từ đầu** hai lần. Và với số `3`, ta lại đếm lại những số `< 2` đã đếm cho số `2`. Công việc lặp lại nhiều — nếu dữ liệu được **sắp xếp**, ta có thể tái sử dụng.

In [4]:
def count_comparisons(n):
    return n * n

for n in [5, 100, 500, 10_000]:
    print(f"n={n:>6}: {count_comparisons(n):>12,} phép so sánh")

n=     5:           25 phép so sánh
n=   100:       10,000 phép so sánh
n=   500:      250,000 phép so sánh
n= 10000:  100,000,000 phép so sánh


`f"{x:,}"` in số có dấu phẩy phân tách hàng nghìn — dễ đọc số lớn.

## 3. Insight — sắp xếp biến "đếm" thành "tìm vị trí"

Sắp xếp `[8, 1, 2, 2, 3]` → `[1, 2, 2, 3, 8]`:

| Chỉ số | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| Giá trị | 1 | 2 | 2 | 3 | 8 |

**Quan sát:** chỉ số **lần xuất hiện đầu tiên** của `x` trong mảng đã sắp = số phần tử đứng trước nó = số phần tử **nhỏ hơn hẳn** `x`.
- `1` xuất hiện đầu tiên ở chỉ số 0 → 0 số nhỏ hơn.
- `2` xuất hiện đầu tiên ở chỉ số 1 → 1 số nhỏ hơn (lần thứ hai ở chỉ số 2 **không** dùng).
- `3` → 3, `8` → 4.

Vì sao phải lấy lần **đầu tiên**? Những số đứng trước lần đầu tiên của `x` đều `< x`; còn sau đó là các bản sao bằng `x`, không được tính.

In [5]:
s = sorted(nums1)
print("Đã sắp:", s)
for x in [1, 2, 3, 8]:
    print(f"x={x}: s.index(x) = {s.index(x)}")

Đã sắp: [1, 2, 2, 3, 8]
x=1: s.index(x) = 0
x=2: s.index(x) = 1
x=3: s.index(x) = 3
x=8: s.index(x) = 4


`list.index(x)` trả chỉ số **đầu tiên** của `x` — đúng thứ ta cần, nhưng nó **duyệt tuyến tính** (`O(n)`) mỗi lần gọi → gọi `n` lần lại thành `O(n²)`. Ta cần tra cứu nhanh hơn → dùng **dict**.

## 4. Nền tảng: `dict` và `enumerate`

- `dict` lưu cặp **key → value**, tra `d[key]` trung bình `O(1)`.
- `key in d` kiểm tra key đã có chưa.
- `enumerate(lst)` sinh cặp `(chỉ số, giá trị)`.

Ta muốn: `first_index[giá trị] = chỉ số đầu tiên trong mảng đã sắp`. Duyệt mảng đã sắp từ trái sang phải, **chỉ lưu khi giá trị chưa có** trong dict.

In [6]:
for i, v in enumerate([1, 2, 2, 3, 8]):
    print(i, v)

first_index = {}
for i, v in enumerate([1, 2, 2, 3, 8]):
    if v not in first_index:     # chỉ lưu lần đầu
        first_index[v] = i
print(first_index)

0 1
1 2
2 2
3 3
4 8
{1: 0, 2: 1, 3: 3, 8: 4}


Mẹo khác: duyệt **ngược** (từ phải sang trái) và **luôn ghi đè** — giá trị cuối cùng được ghi chính là chỉ số nhỏ nhất. Hoặc dùng `d.setdefault(v, i)`: chỉ gán khi `v` chưa có.

In [7]:
s = [1, 2, 2, 3, 8]
d_rev = {}
for i in range(len(s) - 1, -1, -1):     # i = 4, 3, 2, 1, 0
    d_rev[s[i]] = i
print("Duyệt ngược:", d_rev)

d_set = {}
for i, v in enumerate(s):
    d_set.setdefault(v, i)
print("setdefault :", d_set)

Duyệt ngược: {8: 4, 3: 3, 2: 1, 1: 0}
setdefault : {1: 0, 2: 1, 3: 3, 8: 4}


`range(len(s) - 1, -1, -1)`: bắt đầu từ `len(s)-1`, dừng **trước** `-1` (tức dừng ở 0), bước `-1` (đi lùi).

## 5. Cách 2 — Sắp xếp + dict chỉ số đầu tiên

1. `s = sorted(nums)` — bản sao đã sắp (giữ nguyên `nums` để còn trả đúng thứ tự).
2. Dựng `first_index` từ `s`.
3. Kết quả: `[first_index[x] for x in nums]`.

In [8]:
def smaller_sort(nums):
    s = sorted(nums)
    first_index = {}
    for i, v in enumerate(s):
        if v not in first_index:
            first_index[v] = i
    return [first_index[x] for x in nums]

print(smaller_sort(nums1))
print(smaller_sort(nums2))
print(smaller_sort(nums3))

[4, 0, 1, 1, 3]
[2, 1, 0, 3]
[0, 0, 0, 0]


### 5.1. Trace Cách 2 trên `[8, 1, 2, 2, 3]`

| i (trong s) | v | v đã có? | first_index sau bước |
|---|---|---|---|
| 0 | 1 | chưa | {1:0} |
| 1 | 2 | chưa | {1:0, 2:1} |
| 2 | 2 | **có** → bỏ qua | {1:0, 2:1} |
| 3 | 3 | chưa | {1:0, 2:1, 3:3} |
| 4 | 8 | chưa | {1:0, 2:1, 3:3, 8:4} |

Tra theo `nums` gốc: `8→4, 1→0, 2→1, 2→1, 3→3` → `[4, 0, 1, 1, 3]`.

In [9]:
def trace_smaller_sort(nums):
    s = sorted(nums)
    print("nums  =", nums)
    print("sorted=", s)
    first_index = {}
    for i, v in enumerate(s):
        if v not in first_index:
            first_index[v] = i
            note = f"lưu {v} → {i}"
        else:
            note = f"{v} đã có (chỉ số {first_index[v]}), bỏ qua"
        print(f"  i={i}, v={v}: {note:<30} | {first_index}")
    result = [first_index[x] for x in nums]
    print("Kết quả:", result)
    return result

trace_smaller_sort([8, 1, 2, 2, 3])

nums  = [8, 1, 2, 2, 3]
sorted= [1, 2, 2, 3, 8]
  i=0, v=1: lưu 1 → 0                      | {1: 0}
  i=1, v=2: lưu 2 → 1                      | {1: 0, 2: 1}
  i=2, v=2: 2 đã có (chỉ số 1), bỏ qua     | {1: 0, 2: 1}
  i=3, v=3: lưu 3 → 3                      | {1: 0, 2: 1, 3: 3}
  i=4, v=8: lưu 8 → 4                      | {1: 0, 2: 1, 3: 3, 8: 4}
Kết quả: [4, 0, 1, 1, 3]


[4, 0, 1, 1, 3]

## 6. Cách 3 — `bisect_left` trên mảng đã sắp

`bisect_left(s, x)` (module chuẩn `bisect`) dùng **binary search** trả về vị trí nhỏ nhất để chèn `x` vào `s` mà vẫn giữ thứ tự — chính là **số phần tử `< x`**. Không cần dict.

In [10]:
from bisect import bisect_left, bisect_right

s = [1, 2, 2, 3, 8]
for x in [0, 1, 2, 3, 5, 8, 9]:
    print(f"x={x}: bisect_left={bisect_left(s, x)}, bisect_right={bisect_right(s, x)}")

x=0: bisect_left=0, bisect_right=0
x=1: bisect_left=0, bisect_right=1
x=2: bisect_left=1, bisect_right=3
x=3: bisect_left=3, bisect_right=4
x=5: bisect_left=4, bisect_right=4
x=8: bisect_left=4, bisect_right=5
x=9: bisect_left=5, bisect_right=5


- `bisect_left(s, x)` = số phần tử `< x`.
- `bisect_right(s, x)` = số phần tử `<= x`.

Hiệu hai giá trị = số lần `x` xuất hiện.

In [11]:
def smaller_bisect(nums):
    s = sorted(nums)
    return [bisect_left(s, x) for x in nums]

print(smaller_bisect(nums1), smaller_bisect(nums2), smaller_bisect(nums3))

[4, 0, 1, 1, 3] [2, 1, 0, 3] [0, 0, 0, 0]


## 7. Cách 4 — Đếm tần suất + cộng dồn (counting sort) `O(n + V)`

Giá trị chỉ trong `[0, 100]` → dùng mảng `count` dài 101:
1. `count[v]` = số lần `v` xuất hiện.
2. **Cộng dồn**: `smaller[v]` = số phần tử có giá trị `< v` = `count[0] + count[1] + ... + count[v-1]`.
3. Kết quả: `[smaller[x] for x in nums]`.

**Prefix sum (tổng tiền tố):** `prefix[k] = a[0] + ... + a[k-1]`, tính bằng `prefix[k+1] = prefix[k] + a[k]` — một vòng lặp, không cần cộng lại từ đầu.

In [12]:
a = [3, 1, 4, 1, 5]
prefix = [0]
for x in a:
    prefix.append(prefix[-1] + x)
print("a      =", a)
print("prefix =", prefix)       # prefix[k] = tổng a[0..k-1]
print("Tổng a[0..2] =", prefix[3])

a      = [3, 1, 4, 1, 5]
prefix = [0, 3, 4, 8, 9, 14]
Tổng a[0..2] = 8


In [13]:
def smaller_counting(nums):
    count = [0] * 101
    for x in nums:
        count[x] += 1
    smaller = [0] * 101            # smaller[v] = số phần tử < v
    for v in range(1, 101):
        smaller[v] = smaller[v - 1] + count[v - 1]
    return [smaller[x] for x in nums]

print(smaller_counting(nums1), smaller_counting(nums2), smaller_counting(nums3))

[4, 0, 1, 1, 3] [2, 1, 0, 3] [0, 0, 0, 0]


## 8. TRACE Cách 4 trên `[8, 1, 2, 2, 3]`

Chỉ in các giá trị 0..8 (phần còn lại không thay đổi):

| v | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|---|---|---|
| count[v] | 0 | 1 | 2 | 1 | 0 | 0 | 0 | 0 | 1 |
| smaller[v] | 0 | 0 | 1 | 3 | 4 | 4 | 4 | 4 | 4 |

Ví dụ `smaller[3] = smaller[2] + count[2] = 1 + 2 = 3` (các số `< 3` là `1, 2, 2`).

In [14]:
def trace_smaller_counting(nums, show_upto=None):
    if show_upto is None:
        show_upto = max(nums)
    count = [0] * 101
    for x in nums:
        count[x] += 1
    print("count  :", count[: show_upto + 1])
    smaller = [0] * 101
    for v in range(1, 101):
        smaller[v] = smaller[v - 1] + count[v - 1]
        if v <= show_upto:
            print(f"  smaller[{v}] = smaller[{v-1}] + count[{v-1}] = {smaller[v-1]} + {count[v-1]} = {smaller[v]}")
    result = [smaller[x] for x in nums]
    for x in nums:
        print(f"  nums có {x} → smaller[{x}] = {smaller[x]}")
    print("Kết quả:", result)
    return result

trace_smaller_counting([8, 1, 2, 2, 3])

count  : [0, 1, 2, 1, 0, 0, 0, 0, 1]
  smaller[1] = smaller[0] + count[0] = 0 + 0 = 0
  smaller[2] = smaller[1] + count[1] = 0 + 1 = 1
  smaller[3] = smaller[2] + count[2] = 1 + 2 = 3
  smaller[4] = smaller[3] + count[3] = 3 + 1 = 4
  smaller[5] = smaller[4] + count[4] = 4 + 0 = 4
  smaller[6] = smaller[5] + count[5] = 4 + 0 = 4
  smaller[7] = smaller[6] + count[6] = 4 + 0 = 4
  smaller[8] = smaller[7] + count[7] = 4 + 0 = 4
  nums có 8 → smaller[8] = 4
  nums có 1 → smaller[1] = 0
  nums có 2 → smaller[2] = 1
  nums có 2 → smaller[2] = 1
  nums có 3 → smaller[3] = 3
Kết quả: [4, 0, 1, 1, 3]


[4, 0, 1, 1, 3]

## 9. Code nộp LeetCode

Bản counting (`O(n + 101)`):

In [15]:
class Solution:
    def smallerNumbersThanCurrent(self, nums: list[int]) -> list[int]:
        count = [0] * 101
        for x in nums:
            count[x] += 1
        smaller = [0] * 101
        for v in range(1, 101):
            smaller[v] = smaller[v - 1] + count[v - 1]
        return [smaller[x] for x in nums]

sol = Solution()
print(sol.smallerNumbersThanCurrent([8,1,2,2,3]))   # [4,0,1,1,3]
print(sol.smallerNumbersThanCurrent([6,5,4,8]))     # [2,1,0,3]
print(sol.smallerNumbersThanCurrent([7,7,7,7]))     # [0,0,0,0]

[4, 0, 1, 1, 3]
[2, 1, 0, 3]
[0, 0, 0, 0]


Bản sort + dict (không phụ thuộc miền giá trị nhỏ — dùng được khi giá trị lên tới `10^9`):

In [16]:
class SolutionSort:
    def smallerNumbersThanCurrent(self, nums: list[int]) -> list[int]:
        first_index = {}
        for i, v in enumerate(sorted(nums)):
            if v not in first_index:
                first_index[v] = i
        return [first_index[x] for x in nums]

print(SolutionSort().smallerNumbersThanCurrent([8,1,2,2,3]))

[4, 0, 1, 1, 3]


## 10. Vì sao đúng?

**Cách 2 (sort + chỉ số đầu tiên):** trong mảng tăng dần `s`, gọi `k` là chỉ số đầu tiên có `s[k] == x`. Mọi `s[i]` với `i < k` thoả `s[i] <= x` và `s[i] != x` (vì `k` là lần đầu) → `s[i] < x`. Mọi `s[i]` với `i >= k` thoả `s[i] >= x`. Vậy có **đúng `k`** phần tử `< x`. Sắp xếp chỉ đổi thứ tự, không đổi tập giá trị → đếm trên `s` bằng đếm trên `nums`.

**Cách 4 (counting):** invariant sau bước `v` của vòng cộng dồn: `smaller[v] = Σ_{u < v} count[u]` = số phần tử có giá trị `< v`. Ban đầu `smaller[0] = 0` (không có giá trị âm). Bước `v`: `smaller[v] = smaller[v-1] + count[v-1]` cộng thêm đúng các phần tử bằng `v-1` → invariant giữ.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`, `V = 101` (kích thước miền giá trị).

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n²)` | `O(1)` | dễ nhất |
| 2. Sort + dict | `O(n log n)` | `O(n)` | không phụ thuộc miền giá trị |
| 3. Sort + bisect | `O(n log n)` | `O(n)` | gọn |
| 4. Counting + prefix | `O(n + V)` | `O(V)` | tối ưu khi `V` nhỏ |

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [17]:
import random

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.smallerNumbersThanCurrent([8,1,2,2,3]), [4,0,1,1,3])
check("đề 2", sol.smallerNumbersThanCurrent([6,5,4,8]), [2,1,0,3])
check("đề 3 (toàn bằng)", sol.smallerNumbersThanCurrent([7,7,7,7]), [0,0,0,0])
check("hai phần tử", sol.smallerNumbersThanCurrent([5, 0]), [1, 0])
check("biên 0 và 100", sol.smallerNumbersThanCurrent([100, 0, 100, 0]), [2, 0, 2, 0])
check("giảm dần", sol.smallerNumbersThanCurrent([4, 3, 2, 1]), [3, 2, 1, 0])
check("tăng dần", sol.smallerNumbersThanCurrent([1, 2, 3, 4]), [0, 1, 2, 3])

PASS đề 1: [4, 0, 1, 1, 3]
PASS đề 2: [2, 1, 0, 3]
PASS đề 3 (toàn bằng): [0, 0, 0, 0]
PASS hai phần tử: [1, 0]
PASS biên 0 và 100: [2, 0, 2, 0]
PASS giảm dần: [3, 2, 1, 0]
PASS tăng dần: [0, 1, 2, 3]


In [18]:
methods = [smaller_brute, smaller_brute_short, smaller_sort, smaller_bisect, smaller_counting,
           Solution().smallerNumbersThanCurrent, SolutionSort().smallerNumbersThanCurrent]
rng = random.Random(1365)
for _ in range(400):
    nums = [rng.randint(0, 100) for _ in range(rng.randint(2, 50))]
    expected = smaller_brute(nums)
    for f in methods:
        assert f(list(nums)) == expected, f
# mảng lớn nhất theo đề
big = [rng.randint(0, 100) for _ in range(500)]
assert smaller_counting(big) == smaller_brute(big)
print(f"400 mảng ngẫu nhiên × {len(methods)} cách + mảng 500 phần tử: PASS")

400 mảng ngẫu nhiên × 7 cách + mảng 500 phần tử: PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` thay vì `<`
Đếm cả các số **bằng** `x` (kể cả chính nó) → sai.

In [19]:
def wrong_le(nums):
    return [sum(v <= x for v in nums) for x in nums]

print("Sai :", wrong_le([7, 7, 7, 7]), "| Đúng:", smaller_brute([7, 7, 7, 7]))

Sai : [4, 4, 4, 4] | Đúng: [0, 0, 0, 0]


### Lỗi 2: Dùng chỉ số **cuối cùng** trong mảng đã sắp
Ghi đè dict khi duyệt xuôi → mỗi số trùng nhận chỉ số **lần xuất hiện cuối**.

In [20]:
def wrong_last_index(nums):
    d = {}
    for i, v in enumerate(sorted(nums)):
        d[v] = i                      # ghi đè → chỉ số cuối
    return [d[x] for x in nums]

print("Sai :", wrong_last_index([8, 1, 2, 2, 3]), "| Đúng:", smaller_sort([8, 1, 2, 2, 3]))

Sai :

 [4, 0, 2, 2, 3] | Đúng: [4, 0, 1, 1, 3]


### Lỗi 3: Sắp xếp **tại chỗ** rồi trả kết quả theo thứ tự đã sắp
`nums.sort()` làm mất thứ tự gốc; kết quả phải theo thứ tự của `nums` ban đầu.

In [21]:
def wrong_inplace(nums):
    nums.sort()                        # mất thứ tự gốc!
    d = {}
    for i, v in enumerate(nums):
        d.setdefault(v, i)
    return [d[x] for x in nums]

print("Sai :", wrong_inplace([8, 1, 2, 2, 3]), "| Đúng:", smaller_sort([8, 1, 2, 2, 3]))

Sai : [0, 1, 1, 3, 4] | Đúng: [4, 0, 1, 1, 3]


### Lỗi 4: Lệch một ô khi cộng dồn
```python
smaller[v] = smaller[v - 1] + count[v]   # SAI: cộng cả số bằng v
```

In [22]:
def wrong_prefix(nums):
    count = [0] * 101
    for x in nums:
        count[x] += 1
    smaller = [0] * 101
    smaller[0] = count[0]
    for v in range(1, 101):
        smaller[v] = smaller[v - 1] + count[v]   # thành "số phần tử <= v"
    return [smaller[x] for x in nums]

print("Sai :", wrong_prefix([8, 1, 2, 2, 3]), "| Đúng:", smaller_counting([8, 1, 2, 2, 3]))

Sai : [5, 1, 3, 3, 4] | Đúng: [4, 0, 1, 1, 3]


Mẹo sửa: luôn viết rõ **định nghĩa** `smaller[v]` (số phần tử `< v` hay `<= v`?) thành comment trước khi code, rồi kiểm tra với `v = 0`.

### Mẹo nhận dạng pattern

- "Với **mỗi** phần tử, đếm bao nhiêu phần tử nhỏ hơn/lớn hơn nó" → **sort** + vị trí (hoặc `bisect`).
- Miền giá trị nhỏ (`0..100`, chữ cái `a..z`) → **đếm tần suất** + **prefix sum**.
- Cần giữ thứ tự gốc khi trả kết quả → **sắp bản sao** (`sorted`) hoặc sắp **chỉ số**.
- Nếu đề yêu cầu "số nhỏ hơn **ở bên phải**" (LeetCode 315) → khó hơn nhiều: merge sort / Fenwick tree.
- Bài anh em: *Rank Transform of an Array* (1331), *Relative Ranks* (506), *Count of Smaller Numbers After Self* (315).

## 14. Biến thể

### 14.1. Đếm số **lớn hơn** mỗi phần tử
Số phần tử `> x` = `n - bisect_right(s, x)`.

In [23]:
def greater_counts(nums):
    s = sorted(nums)
    n = len(nums)
    return [n - bisect_right(s, x) for x in nums]

print(greater_counts([8, 1, 2, 2, 3]))   # [0, 4, 2, 2, 1]

[0, 4, 2, 2, 1]


### 14.2. Rank Transform (LeetCode 1331)
Hạng của `x` = 1 + số **giá trị phân biệt** nhỏ hơn `x`. Khác bài gốc: số trùng chỉ tính một lần.

In [24]:
def rank_transform(arr):
    rank = {}
    for v in sorted(set(arr)):          # set: bỏ trùng
        rank[v] = len(rank) + 1
    return [rank[x] for x in arr]

print(set([40, 10, 20, 30, 10]))
print(rank_transform([40, 10, 20, 30, 10]))   # [4, 1, 2, 3, 1]

{40, 10, 20, 30}
[4, 1, 2, 3, 1]


`set(...)` tạo **tập hợp**: không có phần tử trùng, thứ tự không đảm bảo → sắp lại bằng `sorted`.

### 14.3. Sắp **chỉ số** thay vì giá trị
Một cách khác giữ thứ tự gốc: sắp danh sách chỉ số `idx` theo `nums[idx]`, rồi điền kết quả vào đúng chỗ.

In [25]:
def smaller_by_index_sort(nums):
    n = len(nums)
    order = sorted(range(n), key=lambda i: nums[i])
    result = [0] * n
    for pos, i in enumerate(order):
        if pos > 0 and nums[i] == nums[order[pos - 1]]:
            result[i] = result[order[pos - 1]]   # số trùng: dùng lại đáp án trước
        else:
            result[i] = pos
    return result

print(smaller_by_index_sort([8, 1, 2, 2, 3]))

[4, 0, 1, 1, 3]


## 15. Bài tập (TODO — đáp án ở cuối)

### Bài 1 — Tính tay
Với `nums = [5, 0, 10, 0, 10, 3]`, hãy tự lập bảng `count` và `smaller` (cho các giá trị xuất hiện) rồi suy ra kết quả.

### Bài 2 — Brute force
Viết lại brute force bằng hai vòng `for` tường minh.

In [26]:
def exercise_brute(nums):
    # TODO
    pass

# Gợi ý test:
# assert exercise_brute([8, 1, 2, 2, 3]) == [4, 0, 1, 1, 3]

### Bài 3 — Sort + dict

In [27]:
def exercise_sort_dict(nums):
    # TODO 1: s = sorted(nums)
    # TODO 2: dict chỉ số đầu tiên
    # TODO 3: tra theo nums gốc
    pass

# Gợi ý test:
# assert exercise_sort_dict([6, 5, 4, 8]) == [2, 1, 0, 3]

### Bài 4 — Counting cho chữ cái
Cho chuỗi `s` chữ thường, trả về list: với mỗi ký tự, có bao nhiêu ký tự trong `s` **nhỏ hơn** nó theo bảng chữ cái.

In [28]:
def exercise_smaller_letters(s):
    # TODO: count dài 26, ord(ch) - ord('a')
    pass

# Gợi ý test:
# assert exercise_smaller_letters("cab") == [2, 0, 1]
# assert exercise_smaller_letters("zzaa") == [2, 2, 0, 0]

### Bài 5 — Số phần tử nằm trong khoảng
Cho `nums` và danh sách truy vấn `(lo, hi)`, với mỗi truy vấn trả số phần tử `x` thoả `lo <= x <= hi`. Giá trị trong `[0, 100]`. Mỗi truy vấn phải `O(1)`.

In [29]:
def exercise_range_counts(nums, queries):
    # TODO: prefix sum trên count
    pass

# Gợi ý test:
# assert exercise_range_counts([8, 1, 2, 2, 3], [(2, 3), (0, 100), (4, 7)]) == [3, 5, 0]

### Bài 6 — Mini challenge: giá trị lên tới `10^9`
Không dùng mảng `count` cỡ miền giá trị. Viết lời giải `O(n log n)` **không** dùng `bisect` và **không** dùng dict (gợi ý: sắp chỉ số).

In [30]:
def exercise_big_values(nums):
    # TODO
    pass

# Gợi ý test:
# assert exercise_big_values([10**9, 1, 10**9, 7]) == [2, 0, 2, 1]

## 16. Gợi ý 3 mức

**Bài 1** — (1) Đếm từng giá trị. (2) `count[0]=2, count[3]=1, count[5]=1, count[10]=2`. (3) `smaller[0]=0, smaller[3]=2, smaller[5]=3, smaller[10]=4`.

**Bài 2** — (1) Vòng ngoài theo `i`, vòng trong theo `j`. (2) Đếm khi `nums[j] < nums[i]`. (3) `result.append(count)` sau vòng trong.

**Bài 3** — (1) Chỉ số đầu tiên trong mảng sắp. (2) `if v not in d: d[v] = i`. (3) `return [d[x] for x in nums]`.

**Bài 4** — (1) Giống bài gốc với miền 26. (2) `count[ord(ch) - 97] += 1`, rồi prefix. (3) Kết quả `smaller[ord(ch) - 97]`.

**Bài 5** — (1) Số phần tử trong `[lo, hi]` = (số `<= hi`) − (số `< lo`). (2) `prefix[v]` = số phần tử `< v`, dài 102. (3) `prefix[hi + 1] - prefix[lo]`.

**Bài 6** — (1) Mục 14.3. (2) `order = sorted(range(n), key=lambda i: nums[i])`. (3) Số trùng thì dùng lại đáp án của phần tử liền trước trong `order`.

## 17. Đáp án tham khảo

**Bài 1:** kết quả `[3, 0, 4, 0, 4, 2]`.

In [31]:
assert Solution().smallerNumbersThanCurrent([5, 0, 10, 0, 10, 3]) == [3, 0, 4, 0, 4, 2]

def answer_brute(nums):
    result = []
    for i in range(len(nums)):
        c = 0
        for j in range(len(nums)):
            if nums[j] < nums[i]:
                c += 1
        result.append(c)
    return result

def answer_sort_dict(nums):
    d = {}
    for i, v in enumerate(sorted(nums)):
        if v not in d:
            d[v] = i
    return [d[x] for x in nums]

def answer_smaller_letters(s):
    count = [0] * 26
    for ch in s:
        count[ord(ch) - ord("a")] += 1
    smaller = [0] * 26
    for k in range(1, 26):
        smaller[k] = smaller[k - 1] + count[k - 1]
    return [smaller[ord(ch) - ord("a")] for ch in s]

def answer_range_counts(nums, queries):
    count = [0] * 101
    for x in nums:
        count[x] += 1
    prefix = [0] * 102              # prefix[v] = số phần tử < v
    for v in range(101):
        prefix[v + 1] = prefix[v] + count[v]
    return [prefix[hi + 1] - prefix[lo] for lo, hi in queries]

def answer_big_values(nums):
    n = len(nums)
    order = sorted(range(n), key=lambda i: nums[i])
    res = [0] * n
    for pos, i in enumerate(order):
        if pos > 0 and nums[i] == nums[order[pos - 1]]:
            res[i] = res[order[pos - 1]]
        else:
            res[i] = pos
    return res

assert answer_brute([8, 1, 2, 2, 3]) == [4, 0, 1, 1, 3]
assert answer_sort_dict([6, 5, 4, 8]) == [2, 1, 0, 3]
assert answer_smaller_letters("cab") == [2, 0, 1]
assert answer_smaller_letters("zzaa") == [2, 2, 0, 0]
assert answer_range_counts([8, 1, 2, 2, 3], [(2, 3), (0, 100), (4, 7)]) == [3, 5, 0]
assert answer_big_values([10**9, 1, 10**9, 7]) == [2, 0, 2, 1]

rng = random.Random(42)
for _ in range(200):
    nums = [rng.randint(0, 10**9 if rng.random() < 0.5 else 5) for _ in range(rng.randint(2, 30))]
    assert answer_big_values(nums) == answer_brute(nums) == answer_sort_dict(nums)
print("Đáp án bài tập: PASS toàn bộ")

Đáp án bài tập: PASS toàn bộ


## 18. Tổng kết — template nhớ nhanh + checklist

```python
# Miền giá trị nhỏ: đếm + cộng dồn
count = [0] * (MAXV + 1)
for x in nums: count[x] += 1
smaller = [0] * (MAXV + 1)                # smaller[v] = số phần tử < v
for v in range(1, MAXV + 1):
    smaller[v] = smaller[v - 1] + count[v - 1]
ans = [smaller[x] for x in nums]

# Miền giá trị lớn: sort + chỉ số đầu tiên
first = {}
for i, v in enumerate(sorted(nums)):
    first.setdefault(v, i)
ans = [first[x] for x in nums]
```

**Checklist trước khi submit**
- [ ] So sánh `<` nghiêm ngặt (số bằng nhau không tính).
- [ ] Lấy chỉ số **đầu tiên** của giá trị trùng.
- [ ] Không phá thứ tự gốc của `nums` (dùng `sorted`, không `nums.sort()` rồi tra).
- [ ] Mảng đếm đủ dài (`101` cho giá trị `0..100`).
- [ ] Đã test mảng toàn bằng nhau, có `0` và `100`, tăng/giảm dần.